# Digital Twin with the OpenAI Agents SDK

This notebook rebuilds the Lab 3 digital twin with the OpenAI Agents SDK. The twin answers questions about my career using my LinkedIn PDF and summary, and saves the email address of anyone who wants to get in touch using a `@function_tool`. It remembers each visitor's conversation through an SDK `Session` and is served through a Gradio chat. A second evaluator agent checks that every reply stays work-related and sends it back for a rewrite if it doesn't. The SDK now handles the tool JSON, the agent loop, and the conversation history that I wrote by hand in Lab 3.

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pypdf import PdfReader
from pydantic import BaseModel
from agents import (
    Agent, Runner, OpenAIChatCompletionsModel,
    function_tool, set_tracing_disabled,
)
import gradio as gr
import os

In [2]:
load_dotenv(override=True)
set_tracing_disabled(True)   # silences the OPENAI_API_KEY trace warning

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.5-flash-lite", openai_client=gemini_client)

In [ ]:
reader = PdfReader("/Users/hirentrivedi/AGENTIC_AI/agents/1_foundations/twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("/Users/hirentrivedi/AGENTIC_AI/agents/1_foundations/twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print(len(linkedin), len(summary))  # quick check: both should be greater than 0

In [5]:
instructions = f"""
# Your role
You are a digital twin running on a website, chatting with visitors of the website.
You represent the person whose website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:
{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context
Here is a summary of the person's LinkedIn profile so that you can answer questions:
{linkedin}

# Rules
Engage with the user. Be professional and engaging, as if talking to a potential client or future employer.
Avoid answering questions that are not related to the person's career, background, skills and experience;
steer the conversation back to professional topics.
Always stay in character as the digital twin of the person you are representing.

If the user shares their email address or wants to get in touch, use the record_email tool.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
"""

In [6]:
@function_tool
def record_email(email: str) -> str:
    """Record that a user provided their email address so the person can contact them later"""
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email recorded"

In [7]:
twin = Agent(
    name="Digital Twin",
    instructions=instructions,
    model=gemini_model,        # object, no quotes
    tools=[record_email],
)

In [ ]:
result = await Runner.run(twin, "Hi, tell me about yourself. My email is test@testy.com")
print(result.final_output)

In [ ]:
from agents import SQLiteSession

# One session per visitor, keyed by Gradio's session id
sessions = {}

async def chat(message, history, request: gr.Request):
    sid = request.session_hash
    if sid not in sessions:
        sessions[sid] = SQLiteSession(sid)   # in-memory by default

    result = await Runner.run(twin, message, session=sessions[sid])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

In [10]:
class Evaluation(BaseModel):
    is_work_related: bool
    feedback: str

evaluator = Agent(
    name="Evaluator",
    instructions="""You review replies from a digital twin that must only discuss the person's
career, background, skills and experience. Given a user question and the twin's reply, set
is_work_related to true only if the reply is strictly about professional topics (or politely
steers back to them). If not, explain briefly in feedback how to fix it.""",
    model=gemini_model,
    output_type=Evaluation,
)

In [ ]:
async def chat(message, history, request: gr.Request):
    sid = request.session_hash
    if sid not in sessions:
        sessions[sid] = SQLiteSession(sid)
    session = sessions[sid]

    result = await Runner.run(twin, message, session=session)
    reply = result.final_output

    for _ in range(2):  # at most 2 retries
        check = await Runner.run(
            evaluator,
            f"User question: {message}\n\nTwin reply: {reply}",
        )
        evaluation = check.final_output
        if evaluation.is_work_related:
            break
        print("Rejected:", evaluation.feedback)
        retry = await Runner.run(
            twin,
            f"Your previous reply was rejected: {evaluation.feedback}. "
            "Rewrite it to stay strictly professional.",
            session=session,
        )
        reply = retry.final_output

    return reply

gr.ChatInterface(chat).launch(inbrowser=True)